<a href="https://colab.research.google.com/github/2am-dev/PDB_actions/blob/main/5D_MATRIX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TCR-pMHC 5D Contact Matrix Builder

This notebook processes a directory of PDB files of TCR–pMHC complexes and produces a 5‑dimensional binary matrix for each complex, as described in the specification.

- Chains: A (MHCα), B (MHCβ), C (peptide), D (TCRβ), E (TCRα)  
- Matrix element = 1 if CA(TCRβ residue) – CA(TCRα residue) distance < 6.0 Å, else 0  
- Padding to global maximum chain lengths  
- Output: compressed `.npz` files + metadata JSON

In [1]:
!pip install Bio cupy-cuda12x -q

In [2]:
import os
import json
import sys
import warnings
from concurrent.futures import ProcessPoolExecutor, as_completed
from typing import Dict, Tuple, List

import numpy as np
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa

warnings.filterwarnings('ignore')   # suppress Bio.PDB noise

# ------------------------- GPU setup -------------------------
try:
    import cupy as cp
    USE_GPU = True
    print("CuPy found – GPU acceleration ENABLED.")
except ImportError:
    USE_GPU = False
    print("CuPy not installed – falling back to CPU (NumPy).")

CuPy found – GPU acceleration ENABLED.


In [3]:
# ------------------------- Configuration -------------------------
PDB_DIR = "/content/drive/MyDrive/IMIL/Results_for_sharing/TCR_complexes"            # directory containing .pdb files
OUTPUT_DIR = "./output_matrices"   # where matrices & metadata are saved
SAVE_AA_SEQUENCES = True           # also store one‑letter sequences per chain
MAX_WORKERS = 4                    # number of parallel processes for parsing

os.makedirs(OUTPUT_DIR, exist_ok=True)

pdb_files = [os.path.join(PDB_DIR, f) for f in os.listdir(PDB_DIR) if f.endswith('.pdb')]
print(f"Found {len(pdb_files)} PDB files.")

Found 541 PDB files.


In [4]:
# ------------------------- Chain mapping & amino acid lookup -------------------------
CHAIN_ORDER = ['A', 'B', 'C', 'D', 'E']
CHAIN_NAMES = {'A': 'MHC_alpha', 'B': 'MHC_beta', 'C': 'peptide',
               'D': 'TCR_beta', 'E': 'TCR_alpha'}

THREE_TO_ONE = {
    'ALA':'A','CYS':'C','ASP':'D','GLU':'E','PHE':'F','GLY':'G','HIS':'H','ILE':'I',
    'LYS':'K','LEU':'L','MET':'M','ASN':'N','PRO':'P','GLN':'Q','ARG':'R','SER':'S',
    'THR':'T','VAL':'V','TRP':'W','TYR':'Y'
}

In [5]:
# ------------------------- Helper: parse a single PDB file -------------------------
def parse_pdb(pdb_path: str) -> Tuple[str, Dict[str, int], Dict[str, np.ndarray], Dict[str, List[str]]]:
    """
    Parse one PDB file and return:
        pdb_id        : file stem (e.g. '1ao7')
        lengths       : dict chain -> number of residues with CA
        coords        : dict chain -> (N,3) float array of CA coordinates
        sequences     : dict chain -> list of one‑letter amino acid codes (only if SAVE_AA is True)
    Missing chains yield length 0 and empty arrays.
    """
    parser = PDBParser(QUIET=True)
    pdb_id = os.path.splitext(os.path.basename(pdb_path))[0]
    structure = parser.get_structure(pdb_id, pdb_path)
    model = structure[0]

    lengths = {}
    coords = {}
    sequences = {}

    for cid in CHAIN_ORDER:
        chain = model[cid] if model.has_id(cid) else None
        if chain is None:
            lengths[cid] = 0
            coords[cid] = np.empty((0, 3), dtype=np.float32)
            sequences[cid] = []
            continue

        # Collect residues with CA, sorted by (resSeq, iCode)
        residues = []
        for res in chain:
            if not is_aa(res) or 'CA' not in res:
                continue
            ca = res['CA'].get_vector().get_array().astype(np.float32)
            res_id = res.get_id()  # (hetero, resseq, icode)
            resname = res.get_resname().strip()
            residues.append((res_id, ca, resname))

        residues.sort(key=lambda x: (x[0][1], x[0][2]))   # sort by sequence number, then insertion code
        if residues:
            coords_list = [r[1] for r in residues]
            coords[cid] = np.array(coords_list, dtype=np.float32)
            lengths[cid] = len(coords_list)
            sequences[cid] = [THREE_TO_ONE.get(r[2], 'X') for r in residues]
        else:
            coords[cid] = np.empty((0, 3), dtype=np.float32)
            lengths[cid] = 0
            sequences[cid] = []

    return pdb_id, lengths, coords, sequences

In [6]:
# ------------------------- Parallel parsing of all PDB files -------------------------
print("Parsing files in parallel ...")
parsed_data = {}   # pdb_id -> {lengths, coords, sequences}

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Submit all tasks
    future_to_file = {executor.submit(parse_pdb, path): path for path in pdb_files}
    for future in as_completed(future_to_file):
        path = future_to_file[future]
        try:
            pdb_id, lengths, coords, sequences = future.result()
            parsed_data[pdb_id] = {'lengths': lengths, 'coords': coords, 'sequences': sequences}
            print(f"  Parsed {pdb_id}")
        except Exception as e:
            print(f"  ERROR parsing {path}: {e}")

print(f"Successfully parsed {len(parsed_data)} structures.")

Parsing files in parallel ...
  Parsed 2iam
  Parsed 2ak4
  Parsed 1ymm
  Parsed 2f53
  Parsed 2esv
  Parsed 1yc7
  Parsed 2ian
  Parsed 2j8u
  Parsed 2po6
  Parsed 2p5w
  Parsed 2ol3
  Parsed 2z31
  Parsed 2vlk
  Parsed 2nx5
  Parsed 2ypl
  Parsed 1qse
  Parsed 2gj6
  Parsed 2f54
  Parsed 2ckb
  Parsed 1g6r
  Parsed 1qrn
  Parsed 1rc3
  Parsed 1oga
  Parsed 1lp9
  Parsed 1fo0
  Parsed 2bnr
  Parsed 1fyt
  Parsed 1kj2
  Parsed 2bnq
  Parsed 1ypz
  Parsed 1zgl
  Parsed 2e7l
  Parsed 1mwa
  Parsed 1d9k
  Parsed 1mi5
  Parsed 1nam
  Parsed 1bd2
  Parsed 1qsf
  Parsed 9nmw
  Parsed 9rxm
  Parsed 1j8h
  Parsed 9nmx
  Parsed 9qqe
  Parsed 9pbh
  Parsed 9o06
  Parsed 1ao7
  Parsed 9pbg
  Parsed 9wbd
  Parsed 9ru5
  Parsed 9o05
  Parsed 9o08
  Parsed 9rup
  Parsed 9o07
  Parsed 9k2r
  Parsed 9l1l
  Parsed 9nmu
  Parsed 9k2i
  Parsed 9ms0
  Parsed 9nw7
  Parsed 9nmv
  Parsed 9nig
  Parsed 9nw6
  Parsed 9nw8
  Parsed 9nds
  Parsed 9nw9
  Parsed 3o8x
  Parsed 9nmy
  Parsed 3he6
  Parsed 3gsn
  Pa

In [7]:
# ------------------------- Global maximum lengths -------------------------
max_lengths = {cid: 0 for cid in CHAIN_ORDER}
for data in parsed_data.values():
    for cid in CHAIN_ORDER:
        if data['lengths'][cid] > max_lengths[cid]:
            max_lengths[cid] = data['lengths'][cid]

print("\nGlobal maximum chain lengths:")
for cid in CHAIN_ORDER:
    print(f"  {cid} ({CHAIN_NAMES[cid]}): {max_lengths[cid]}")


Global maximum chain lengths:
  A (MHC_alpha): 182
  B (MHC_beta): 95
  C (peptide): 20
  D (TCR_beta): 122
  E (TCR_alpha): 124


In [8]:
# ------------------------- Sanity check: dense matrix size -------------------------
shape = tuple(max_lengths[c] for c in CHAIN_ORDER)
total_elements = np.prod(shape)
print(f"\n5D matrix shape : {shape}")
print(f"Total elements  : {total_elements:,}")

# Warn if the product exceeds ~200 million elements (about 1.6 GB as uint8)
FEASIBLE_THRESHOLD = 200_000_000
if total_elements > FEASIBLE_THRESHOLD:
    print("\n⚠️  WARNING: The requested dense 5D matrix is extremely large "
          "(>200 million elements).")
    print("   This may exceed available RAM and is not recommended. "
          "Consider using a sparse format or trimming chain lengths.")
    if total_elements > 1e10:
        print("\n❌  ABORTING: Matrix size completely infeasible "
              f"({total_elements/1e9:.1f} billion elements).")
        sys.exit(1)


5D matrix shape : (182, 95, 20, 122, 124)
Total elements  : 5,231,262,400

⚠️  WARNING: The requested dense 5D matrix is extremely large (>200 million elements).
   This may exceed available RAM and is not recommended. Consider using a sparse format or trimming chain lengths.


In [ ]:
# ------------------------- Build and save matrices (GPU used for distance) -------------------------
metadata = {
    'chain_mapping': CHAIN_NAMES,
    'max_lengths': max_lengths,
    'complexes': {}
}
aa_identities = {} if SAVE_AA_SEQUENCES else None

print("\nBuilding 5D matrices and saving ...")
for pdb_id, data in parsed_data.items():
    lengths = data['lengths']
    coords = data['coords']
    metadata['complexes'][pdb_id] = {'actual_lengths': lengths}
    if SAVE_AA_SEQUENCES:
        aa_identities[pdb_id] = {cid: data['sequences'][cid] for cid in CHAIN_ORDER}

    # ----- Compute D‑E contact map -----
    coords_D = coords['D']
    coords_E = coords['E']
    lenD, lenE = lengths['D'], lengths['E']

    if lenD == 0 or lenE == 0:
        contact_map = np.zeros((lenD, lenE), dtype=bool)
    else:
        if USE_GPU:
            # GPU accelerated distance computation
            D_gpu = cp.asarray(coords_D)
            E_gpu = cp.asarray(coords_E)
            diff = D_gpu[:, None, :] - E_gpu[None, :, :]      # (lenD, lenE, 3)
            dist = cp.sqrt((diff ** 2).sum(axis=2))
            contact_map = cp.asnumpy(dist < 6.0)              # boolean, move back to CPU
        else:
            # CPU fallback
            diff = coords_D[:, None, :] - coords_E[None, :, :]
            dist = np.sqrt((diff ** 2).sum(axis=2))
            contact_map = dist < 6.0                          # (lenD, lenE) bool

    # ----- Build padded 5D matrix on CPU (to avoid GPU memory blow‑up) -----
    mat = np.zeros(tuple(max_lengths[c] for c in CHAIN_ORDER), dtype=np.uint8)
    lenA, lenB, lenC = lengths['A'], lengths['B'], lengths['C']
    # Insert the 2D contact map into the full 5D space using broadcasting
    # We set mat[i,j,k,l,m] = 1 only where contact_map[l,m] is True and within actual lengths
    mat[:lenA, :lenB, :lenC, :lenD, :lenE] = contact_map[np.newaxis, np.newaxis, np.newaxis, :, :]

    # ----- Save compressed matrix -----
    out_path = os.path.join(OUTPUT_DIR, f"{pdb_id}_contact_matrix.npz")
    np.savez_compressed(out_path, matrix=mat)
    print(f"  Saved {pdb_id}   shape: {mat.shape}   non‑zero: {mat.sum()}")

# ----- Save global metadata and optional amino acid sequences -----
meta_path = os.path.join(OUTPUT_DIR, "complex_metadata.json")
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"\nMetadata written to {meta_path}")

if SAVE_AA_SEQUENCES:
    seq_path = os.path.join(OUTPUT_DIR, "aa_identities.json")
    with open(seq_path, 'w') as f:
        json.dump(aa_identities, f, indent=2)
    print(f"Amino‑acid identities written to {seq_path}")


Building 5D matrices and saving ...
  Saved 2iam   shape: (182, 95, 20, 122, 124)   non‑zero: 225990
  Saved 2ak4   shape: (182, 95, 20, 122, 124)   non‑zero: 0
  Saved 1ymm   shape: (182, 95, 20, 122, 124)   non‑zero: 313404
  Saved 2f53   shape: (182, 95, 20, 122, 124)   non‑zero: 0
  Saved 2esv   shape: (182, 95, 20, 122, 124)   non‑zero: 0
  Saved 1yc7   shape: (182, 95, 20, 122, 124)   non‑zero: 0
  Saved 2ian   shape: (182, 95, 20, 122, 124)   non‑zero: 327600
  Saved 2j8u   shape: (182, 95, 20, 122, 124)   non‑zero: 0
  Saved 2po6   shape: (182, 95, 20, 122, 124)   non‑zero: 0
  Saved 2p5w   shape: (182, 95, 20, 122, 124)   non‑zero: 0
  Saved 2ol3   shape: (182, 95, 20, 122, 124)   non‑zero: 0
  Saved 2z31   shape: (182, 95, 20, 122, 124)   non‑zero: 1093092
  Saved 2vlk   shape: (182, 95, 20, 122, 124)   non‑zero: 0
  Saved 2nx5   shape: (182, 95, 20, 122, 124)   non‑zero: 0
  Saved 2ypl   shape: (182, 95, 20, 122, 124)   non‑zero: 0
  Saved 1qse   shape: (182, 95, 20, 122, 1

## Notes

- **Parallelism**: Parsing of all PDB files is done in parallel with `MAX_WORKERS` processes.  
- **GPU acceleration**: If CuPy is present, the pairwise distance matrix for chains D and E is computed on the GPU. The large 5D matrix itself is always built on the CPU to avoid GPU memory limits.  
- **Memory warning**: A threshold of 200 million elements is set. Above that a warning appears, and above 10 billion the script stops to prevent system lock‑up. Adjust `FEASIBLE_THRESHOLD` if your hardware allows more.
- **Output**: For each PDB, `{pdb_id}_contact_matrix.npz` contains a 5D `uint8` array. Metadata (`complex_metadata.json`) maps chain identifiers, maximum lengths, and actual lengths. If `SAVE_AA_SEQUENCES=True`, amino acid identities are stored in `aa_identities.json`.

In [ ]:
# Cell: Find complexes with peptide (chain C) longer than 30 residues
threshold = 30
print(f"Complexes where chain C length > {threshold}:")
found_any = False

# Use the in-memory parsed_data dictionary if available; otherwise load metadata JSON
if 'parsed_data' in globals():
    for pdb_id, d in parsed_data.items():
        lenC = d['lengths']['C']
        if lenC > threshold:
            print(f"  {pdb_id} : chain C length = {lenC}")
            found_any = True
else:
    # fallback: load the saved metadata
    import json
    with open(os.path.join(OUTPUT_DIR, "complex_metadata.json"), 'r') as f:
        meta = json.load(f)
    for pdb_id, info in meta['complexes'].items():
        lenC = info['actual_lengths']['C']
        if lenC > threshold:
            print(f"  {pdb_id} : chain C length = {lenC}")
            found_any = True

if not found_any:
    print("  None found.")